# DNS Software Analysis via CHAOS TXT Records

We request `TXT CHAOS` records (e.g., `id.server`, `version.bind`) from CAIDA Ark vantage points
to identify the DNS software running on anycast prefixes.

**Key findings:**
- **190 ASes** (out of 658 operating anycast DNS) reveal their software via CHAOS TXT — BIND is the most common, followed by PowerDNS
- **25 ASes** deploy different DNS software across PoPs sharing the same anycast IP, including root server and ccTLD operators

In [3]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent))

from add_ASN import CaidaASLookup
from datetime import datetime

In [4]:
chaosv4 = pd.read_csv('2026-01-16_DNSv4.csv.gz')
chaosv6 = pd.read_csv('2026-01-16_DNSv6.csv.gz')

# Two VPs (blq-it, bwi3-us) hit their local dnsmasq resolver for every target — exclude them
chaosv4 = chaosv4[~chaosv4['vp_name'].isin(['blq-it', 'bwi3-us'])]

## Data Overview

In [5]:
for label, df in [('IPv4', chaosv4), ('IPv6', chaosv6)]:
    print(f'--- {label} ---')
    print(f'  {df["vp_name"].nunique()} VPs -> {df["dst"].nunique()} targets with a CHAOS response')
    for qname in sorted(df['qname'].unique()):
        n = df.loc[df['qname'] == qname, 'dst'].nunique()
        print(f'  {n:>5,} targets support {qname}')
    print()

--- IPv4 ---
  240 VPs -> 2168 targets with a CHAOS response
    350 targets support authors.bind
  2,107 targets support hostname.bind
  1,438 targets support id.server
     29 targets support server.bind
  1,555 targets support version.bind
    424 targets support version.server

--- IPv6 ---
  73 VPs -> 2018 targets with a CHAOS response
    538 targets support authors.bind
  1,824 targets support hostname.bind
  1,133 targets support id.server
     23 targets support server.bind
  1,584 targets support version.bind
    318 targets support version.server



## Software Classification

We classify DNS software from response strings.

In [6]:
chaosv4['software'] = 'unknown'

chaosv4.loc[
    chaosv4['record'].str.lower().str.contains('powerdns', na=False),
    'software'
] = 'powerdns'

chaosv4.loc[
    chaosv4['record'].str.lower().str.contains('knot', na=False),
    'software'
] = 'knot'

chaosv4.loc[
    chaosv4['record'].str.lower().str.contains(
        r'bind|9\.11\.22|michael graff|brian wellington|mark andrews|bob halley|james brister|michael sawyer',
        na=False, regex=True
    ),
    'software'
] = 'bind'

chaosv4.loc[
    chaosv4['record'].str.lower().str.contains('redhat', na=False),
    'software'
] = 'redhat'

chaosv4.loc[
    chaosv4['record'].str.lower().str.contains('ultradns', na=False),
    'software'
] = 'ultradns'

chaosv4.loc[
    chaosv4['record'].str.lower().str.contains('core', na=False),
    'software'
] = 'core'

In [7]:
ts = datetime(2026, 1, 16)
caida = CaidaASLookup(ts)
chaosv4 = caida.add_prefix_and_asn(chaosv4, addr_col='dst', ip_version='v4')

.......... Done.
Processed 1073017 prefixes.
Mapping 1389801 IPs against V4 tree...


ASN Lookup:   0%|          | 0/1389801 [00:00<?, ?it/s]

## DNS Software Distribution

In [8]:
per_sw = chaosv4.groupby('software').agg(
    anycast_ips=('dst', 'nunique'),
    ases=('ASN', 'nunique'),
).sort_values('ases', ascending=False)

known = chaosv4[chaosv4['software'] != 'unknown']

per_sw.loc['_known'] = [known['dst'].nunique(), known['ASN'].nunique()]
per_sw.loc['_total'] = [chaosv4['dst'].nunique(), chaosv4['ASN'].nunique()]

per_sw

,anycast_ips,ases
software,,
unknown,2130,556
bind,356,92
powerdns,199,84
knot,48,26
redhat,23,14
core,9,6
ultradns,91,3
_known,682,190
_total,2168,562


## Multiple DNS Software on the Same Anycast IP

Depending on which VP queries a given anycast address, a **different DNS software** is observed.
This reveals operators running heterogeneous software across PoPs.

In [9]:
dst_sw_count = known.groupby('dst')['software'].nunique()
multi_sw_ips = dst_sw_count[dst_sw_count > 1].index

multi_sw = (
    known[known['dst'].isin(multi_sw_ips)]
    .groupby('dst')['software']
    .agg(lambda s: sorted(s.unique()))
    .reset_index()
    .rename(columns={'software': 'software_seen'})
)

dst_asn = chaosv4[['dst', 'ASN']].drop_duplicates()
multi_sw = multi_sw.merge(dst_asn, on='dst')

print(f'{len(multi_sw)} anycast IPs across {multi_sw["ASN"].nunique()} ASes')
multi_sw.sort_values('ASN')

42 anycast IPs across 26 ASes


,dst,software_seen,ASN
13,192.175.48.1,"[bind, knot]",112
14,192.31.196.1,"[bind, knot]",112
39,78.37.72.203,"[bind, redhat]",12389
0,103.51.60.8,"[bind, redhat]",134074
2,13.248.177.206,"[bind, redhat]",16509
4,15.197.207.2,"[bind, core, redhat]",16509
5,15.197.237.120,"[bind, redhat]",16509
30,3.33.202.173,"[bind, core, redhat]",16509
31,3.33.216.193,"[bind, redhat]",16509
37,76.223.1.101,"[bind, redhat]",16509


In [10]:
# ASes that use multiple software across *any* of their anycast IPs
as_sw_count = known.groupby('ASN')['software'].nunique()
multi_sw_ases = as_sw_count[as_sw_count > 1].sort_values(ascending=False)

print(f'{len(multi_sw_ases)} ASes use more than one DNS software across their anycast infrastructure')
multi_sw_ases

32 ASes use more than one DNS software across their anycast infrastructure


ASN
16509          4
396982         3
112            2
394353         2
8038_203993    2
7311           2
58023          2
55002          2
52306          2
52304          2
52234          2
51975          2
51966          2
49287          2
41346          2
40355_42926    2
34529          2
12389          2
31529          2
30295          2
27866          2
25354          2
25152          2
209622         2
208366         2
202022         2
199524         2
197997         2
197000         2
16913          2
134074         2
8075           2
Name: software, dtype: int64

### Notable examples of multi-software anycast deployments

| Category | Examples |
|----------|----------|
| Root servers | K-root (`194.0.33.53`), B-root (`170.247.171.2`) |
| AS112 project | `192.175.48.1`, `192.31.196.1` |
| Cloud DNS | AWS Route 53 / Global Accelerator, Google (`35.243.187.176`) |
| ccTLD operators | CIRA (.ca), NIC Chile (.cl) |
| RIR infrastructure | LACNIC, RIPE (`193.0.*`), APNIC — rDNS |
| CDN / Hosting | Verizon / Edgecast (`69.31.181.100`), Gcore |